In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
def compare():

    '''
    Plot the EWs and corresponding uncertainties measured by this work against those measured by E24
    '''

    # Establish common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    # Open the E24 catalog
    catalog = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog)

    # Get the object IDs of the E24 catalog
    ids_endsley2024 = hdul_endsley2024[1].data['ID']

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    # Set the string identifiers of the different absolute UV magnitude bins
    m_uv_bins = ['bright', 'faint', 'vfaint']

    # Set the labels for the legends of the figure
    labels_markers = ['with Lya', 'no Lya']

    # Set the colors of the markers for the two sets of fits
    colors = ['orange', 'blue']

    # Instantiate the two figures
    fig, ax = plt.subplots()
    fig_err, ax_err = plt.subplots()

    # For each set of fits
    for i, name in enumerate(names):

        # For each absolute UV magnitude bin
        for j, m_uv_bin in enumerate(m_uv_bins):

            # Open the corresponding file containing the EWs measured in this work
            with h5py.File(f'{results}/ew/{name}_ews_{m_uv_bin}.h5', 'r') as f:

                # For each object ID in the file
                for k, id in enumerate(list(f.keys())):

                    # Get the posterior distribution of the [O III] + H-beta EWs measured in this work, and their corresponding probabilities
                    ew_posterior_me = f[id]['h_beta_ews'][:] + f[id]['o_iii_ews'][:]
                    probs = f[id]['probabilities'][:]

                    # Calculate the probability-weighted median and 16th and 84th percentiles of the [O III] + H-beta EW posterior distribution
                    ew_me, ew_lower, ew_upper = weighted_quantile(ew_posterior_me, probs, [0.5, 0.16, 0.84])

                    # Get the E24 measurement of the median and 16th and 84th percentiles of the [O III] + H-beta EW for this object
                    ew_endsley2024, ew_endsley2024_lower, ew_endsley2024_upper = hdul_endsley2024[1].data['OIIIHbEw'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_l16'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_u84'][ids_endsley2024 == id][0]

                    # Plot the EWs measured in this work against those measured in E24
                    ax.errorbar(ew_me, ew_endsley2024, xerr=[[ew_me - ew_lower], [ew_upper - ew_me]], yerr=[[ew_endsley2024 - ew_endsley2024_lower], [ew_endsley2024_upper - ew_endsley2024]], color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
                    # Plot the uncertainties on the EWs measured in this work against those measured in E24
                    ax_err.scatter(ew_upper - ew_lower, ew_endsley2024_upper - ew_endsley2024_lower, color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
    # Label the axes of each figure
    ax.set_xlabel('EW([O III] + H$\\beta$) (this work)')
    ax.set_ylabel('EW([O III] + H$\\beta$) (E24)')
    ax_err.set_xlabel('P$_{84}$(EW) - P$_{16}$(EW) (this work)')
    ax_err.set_ylabel('P$_{84}$(EW) - P$_{16}$(EW) (E24)')

    # Add the legends to the figures
    ax.legend(loc='upper left')
    ax_err.legend(loc='upper left')

    # Create a one-to-one line on the figures
    ax.axline((1000,1000), slope=1, color='red', linestyle='dashed')
    ax_err.axline((1000,1000), slope=1, color='red', linestyle='dashed')

    # Place both axes on the figures on a logarithmic scale
    ax.loglog()
    ax_err.loglog()

    # Save the figures
    fig.savefig(f'{figs}/compare_ew.png', dpi=200, bbox_inches='tight')
    fig_err.savefig(f'{figs}/compare_ew_errors.png', dpi=200, bbox_inches='tight')

'''
def compare_low_ews():

    # Establish common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    catalog = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog)

    ids_endsley2024 = hdul_endsley2024[1].data['ID']

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    m_uv_bins = ['bright', 'faint', 'vfaint']

    # Set the labels for the legends of the figure
    labels_markers = ['with Lya', 'no Lya']

    # Set the colors of the markers for the two sets of fits
    colors = ['orange', 'blue']

    fig, ax = plt.subplots()
    fig_err, ax_err = plt.subplots()

    for i, name in enumerate(names):

        for j, m_uv_bin in enumerate(m_uv_bins):

            with h5py.File(f'{results}/ew/{name}_ews_{m_uv_bin}.h5', 'r') as f:

                for k, id in enumerate(list(f.keys())):

                    ew_posterior_me = f[id]['h_beta_ews'][:] + f[id]['o_iii_ews'][:]
                    probs = f[id]['probabilities'][:]

                    ew_me, ew_lower, ew_upper = weighted_quantile(ew_posterior_me, probs, [0.5, 0.16, 0.84])

                    ew_endsley2024, ew_endsley2024_lower, ew_endsley2024_upper = hdul_endsley2024[1].data['OIIIHbEw'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_l16'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_u84'][ids_endsley2024 == id][0]

                    ax.errorbar(ew_me, ew_endsley2024, xerr=[[ew_me - ew_lower], [ew_upper - ew_me]], yerr=[[ew_endsley2024 - ew_endsley2024_lower], [ew_endsley2024_upper - ew_endsley2024]], color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
                    ax_err.scatter(ew_upper - ew_lower, ew_endsley2024_upper - ew_endsley2024_lower, color=colors[i], label=f'{labels_markers[i] if j == 0 and k == 0 else ''}', alpha=0.5)
                
    ax.set_xlabel('EW([O III] + H$\\beta$) (this work)')
    ax.set_ylabel('EW([O III] + H$\\beta$) (E24)')

    ax_err.set_xlabel('P$_{84}$(EW) - P$_{16}$(EW) (this work)')
    ax_err.set_ylabel('P$_{84}$(EW) - P$_{16}$(EW) (E24)')

    ax.legend(loc='upper left')
    ax_err.legend(loc='upper left')

    # Create a one-to-one line on the figures
    ax.axline((1000,1000), slope=1, color='red', linestyle='dashed')
    ax_err.axline((1000,1000), slope=1, color='red', linestyle='dashed')

    ax.loglog()
    ax_err.loglog()

    fig.savefig(f'{figs}/compare_ew.png', dpi=200, bbox_inches='tight')
    fig_err.savefig(f'{figs}/compare_ew_errors.png', dpi=200, bbox_inches='tight')
'''

def plot_ews():

    '''
    Plot diagnostic figures of the EW measurements for each line for each object, showing the posterior model SEDs, measured continuum levels, the line and continuum sampling bands, and the measured EWs
    '''

    # Establish common directories
    home = os.getcwd()
    figs = f'{home}/figs'
    results = f'{home}/results'

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    # Create a dictionary of the emission lines to measure the EWs of, and the wavelength bands to measure the line and continuum flux with
    lines = {
        'o_iii' : [r'[O III] 4959, 5007 $\mathrm{\AA}$', [[4954,5011]], [[4880,4930],[5050,5100]]],
        'h_beta' : [r'H$\beta$', [[4856,4866]], [[4775,4825],[4880,4930]]]
    }

    # For each set of fits
    for i, name in enumerate(names):

        # Get a list of the file paths of the BEAGLE fits for this set of fits
        files = glob.glob(f'{results}/beagle_fits/{name}/*_GOODS*_BEAGLE.fits.gz')

        # For each file
        for j, file in enumerate(files):

            # Check if the file is empty; skip if so
            if os.stat(file).st_size == 0:
                print(f'File {os.path.basename(file)} is empty. Skipping.')
                continue

            # Get the ID of the object from the file name
            id = os.path.basename(file).split('.')[0][:-7]

            # Open the HDUL of the file
            hdul = fits.open(file)

            # Get the rest wavelengths of the model SEDs
            w_rest = hdul['FULL SED WL'].data['WL'][0] * u.angstrom

            # Get the rest-frame wavelength-space flux densities of the model SEDs
            seds = hdul['FULL SED'].data * u.erg / u.s / u.cm**2 / u.angstrom

            # Get the associated probabilities of the model SEDs
            probabilities = hdul['POSTERIOR PDF'].data['probability']

            # Make zero-filled arrays to later fill with the EWs and continuum flux densities of each line for each SED
            ews = np.zeros((len(lines), len(seds)), dtype=np.float64) * u.angstrom
            conts = np.zeros((len(lines), len(seds)), dtype=np.float64) * u.erg / u.s / u.cm**2 / u.angstrom

            # For each posterior model SED
            for j, sed in enumerate(seds):

                # Make a false-filled boolean mask of the wavelength bins
                mask = np.zeros(len(w_rest), dtype=bool)

                # For each line
                for k, line in enumerate(lines):

                    # Get the continuum-sampling bands for this line
                    bands_cont = lines[line][2]

                    # Make a false-filled boolean mask of the continuum to sample, to fill in later with true values where the continuum should be sampled
                    mask_cont = np.zeros(len(w_rest), dtype=bool)

                    # For each continuum sapling band
                    for band in bands_cont:

                        # Make the corresponding elements in the continuum mask true
                        mask_cont |= ((w_rest >= band[0] * u.angstrom) & (w_rest <= band[1] * u.angstrom))

                    # Calculate the continuum level as the median flux density in the continuum bands, and add it to the array of measured continuum flux densities
                    line_cont = np.median(sed[mask_cont])
                    conts[k,j] = line_cont
    
                    # Get the line sampling bands for this line
                    bands_line = lines[line][1]

                    # Make a false-filled boolean mask of the line to sample, to fill in later with true values where the line should be sampled
                    mask_line = np.zeros(len(w_rest), dtype=bool)

                    # For each line sampling band
                    for band in bands_line:

                        # Make the corresponding elements in the line mask true
                        mask_line |= ((w_rest >= band[0] * u.angstrom) & (w_rest <= band[1] * u.angstrom))

                    # Calculate the EW of the line by integrating the continuum-normalized flux density over the line sampling bands, and add it to the array of measured EWs
                    ews[k,j] = -1 * np.trapz(1 - (sed / line_cont)[mask_line], w_rest[mask_line], axis=0)

            # For each line
            for j, line in enumerate(lines):

                # Instantiate a diagnostic figure for the EW calculations
                fig, ax = plt.subplots()

                # Calculate the extrema wavelengths between the line and continuum bands
                w_min = np.min([min(lines[line][1]), min(lines[line][2])])
                w_max = np.max([max(lines[line][1]), max(lines[line][2])])

                # Calculate the width of the plot (before padding) based on the minimum and maximum wavelengths of the line and continuum bands
                width = w_max - w_min #np.max([max(lines[line][1]), max(lines[line][2])]) - np.min([min(lines[line][1]), min(lines[line][2])])

                # Create a mask to plot only the region around the line and continuum bands, with some padding on either side
                mask = (w_rest >= (w_min - 0.1 * width) * u.angstrom) & (w_rest <= (w_max + 0.1 * width) * u.angstrom)

                # Calculate the probability-weighted 16th and 84th percentiles of the flux density of the posterior model SEDs at each wavelength in the masked region
                p16 = [weighted_quantile(seds[:,idx].value, probabilities, 0.16) for idx in np.where(mask)[0]]
                p84 = [weighted_quantile(seds[:,idx].value, probabilities, 0.84) for idx in np.where(mask)[0]]

                # Plot a shaded band between the probability-weighted 16th and 84th percentiles of the flux density of the posterior model SEDs at each wavelength in the masked region
                ax.fill_between(w_rest.value[mask], p16, p84, color='red', alpha=0.2)

                # Label the axes
                ax.set_xlabel(r'Rest wavelength ($\mathrm{\AA}$)')
                ax.set_ylabel('Rest-frame flux density (erg s$^{-1}$ cm$^{-2}$ $\mathrm{\AA}^{-1}$)')

                # Set the title of the plot with the BEAGLE ID
                ax.set_title(f'{os.path.basename(file).split('_BEAGLE')[0]}')

                # Calculate the probability-weighted median and 16th and 84th percentiles of the EW of the line
                ew = weighted_quantile(ews.value[j,:], probabilities, 0.5)
                ew_lower = weighted_quantile(ews.value[j,:], probabilities, 0.16)
                ew_upper = weighted_quantile(ews.value[j,:], probabilities, 0.84)

                # Annotate the figure with the name of the line and the EW measurement
                at = AnchoredText(rf'{lines[line][0]}' + f'\n${ew:.0f}_{{-{(ew - ew_lower):.0f}}}^{{+{(ew_upper - ew):.0f}}}$' + r' $\mathrm{\AA}$', 
                    loc='upper right', frameon=False, prop=dict(horizontalalignment='right'))
                ax.add_artist(at)

                # Set the x-axis limits to be slightly wider than the minimum and maximum wavelengths of the line and continuum bands
                ax.set_xlim(w_min - 0.05 * width, w_max + 0.05 * width)

                # For each continuum-sampling band
                for k, band in enumerate(lines[line][2]):

                    # Set the label for the legend only for the first band, to avoid duplicate labels
                    label = 'Continuum band' if k == 0 else None

                    # Plot the continuum-sampling band as a shaded region
                    ax.axvspan(band[0], band[1], alpha=0.2, color='blue', label=label)

                # Plot each line-sampling band
                for k, band in enumerate(lines[line][1]):

                    # Set the label for the legend only for the first band, to avoid duplicate labels
                    label = 'Line band' if k == 0 else None

                    # Plot the continuum-sampling band as a shaded region
                    ax.axvspan(band[0], band[1], alpha=0.2, color='orange', label=label)

                # Plot the probability-weighted median flux density of the continuum as a dashed horizontal line
                ax.axhline(weighted_quantile(conts.value[j], probabilities, 0.5), color='black', ls='dashed')

                # Plot a shaded band between the probability-weighted 16th and 84th percentiles of the continuum flux density estimate
                ax.fill_between(w_rest.value[mask], weighted_quantile(conts.value[j], probabilities, 0.16), weighted_quantile(conts.value[j], probabilities, 0.84), color='black', alpha=0.2)

                # Add the legend to the figure
                ax.legend(loc='upper left')

                # Scale the flux density logarithmically to better show the continuum level and peak of the line
                ax.set_yscale('log')

                # Save the figure
                fig.savefig(f'{figs}/ews/{name}/{id}_{line}.png', dpi=200, bbox_inches='tight')

                # Close the figure to free up memory
                plt.close('all')

def low_ew_ids():

    '''
    Print the object IDs of the sources with low EWs (< 100 angstrom) in E24, which nearly universally have much higher EWs in this work
    '''

    # Establish common directories
    home = os.getcwd()
    data = f'{home}/data'
    results = f'{home}/results'

    catalog = f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits'
    hdul_endsley2024 = fits.open(catalog)

    ids_endsley2024 = hdul_endsley2024[1].data['ID']

    # Set the prefixes of the two sets of fits that this work made (one with and one without Lya but otherwise identical)
    names = ['JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits', 'JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts_beagle_csfh_fits_no_lya']

    m_uv_bins = ['bright', 'faint', 'vfaint']

    for i, name in enumerate(names):

        print(name + '\n')

        for j, m_uv_bin in enumerate(m_uv_bins):

            with h5py.File(f'{results}/ew/{name}_ews_{m_uv_bin}.h5', 'r') as f:

                for k, id in enumerate(list(f.keys())):

                    ew_posterior_h_beta = f[id]['h_beta_ews'][:]
                    ew_posterior_o_iii = f[id]['o_iii_ews'][:]

                    ew_posterior_me = f[id]['h_beta_ews'][:] + f[id]['o_iii_ews'][:]
                    probs = f[id]['probabilities'][:]

                    ew_hbeta, ew_hbeta_lower, ew_hbeta_upper = weighted_quantile(ew_posterior_h_beta, probs, [0.5, 0.16, 0.84])
                    ew_oiii, ew_oiii_lower, ew_oiii_upper = weighted_quantile(ew_posterior_o_iii, probs, [0.5, 0.16, 0.84])

                    ew_me, ew_lower, ew_upper = weighted_quantile(ew_posterior_me, probs, [0.5, 0.16, 0.84])

                    ew_endsley2024, ew_endsley2024_lower, ew_endsley2024_upper = hdul_endsley2024[1].data['OIIIHbEw'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_l16'][ids_endsley2024 == id][0], hdul_endsley2024[1].data['OIIIHbEw_u84'][ids_endsley2024 == id][0]

                    if ew_endsley2024 < 100:
                        print(f'ID {id} has low EW in E24: {ew_endsley2024:.0f} +{ew_endsley2024_upper - ew_endsley2024:.0f} -{ew_endsley2024 - ew_endsley2024_lower:.0f} angstrom,' 
                            + f'\nbut much higher EW in this work: {ew_me:.0f} +{ew_upper - ew_me:.0f} -{ew_me - ew_lower:.0f} angstrom'
                            + f'\nwith Hbeta EW: {ew_hbeta:.0f} +{ew_hbeta_upper - ew_hbeta:.0f} -{ew_hbeta - ew_hbeta_lower:.0f} angstrom'
                            + f'\nand OIII EW: {ew_oiii:.0f} +{ew_oiii_upper - ew_oiii:.0f} -{ew_oiii - ew_oiii_lower:.0f} angstrom\n')

In [ ]:
compare()

In [ ]:
plot_ews()

In [ ]:
low_ew_ids()